# Multi-ToolOrchestration

**Module 10 · Lesson 10.3**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Parallel Execution
- The Task DAG
- The ReAct Loop
- Agent Architectures
- State Management
- Error Handling for Chains

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
!pip install nest_asyncio -q  # noqa
# Uncomment the next line if needed:
# !pip install openai -q

# Reproducibility - the lesson uses seeded random values throughout

## The Trip That Needed Three Tools

> 💡 **Info**
>
> Ask an agent: “find the cheapest GenAI course, add 18% GST, and book it if it’s under my budget.’’ No single tool call can do this. It needs `search` to find the course, THEN `add_gst` on that result (sequential — the second needs the first), THEN a *conditional* branch: if the total is under budget, `book`; otherwise suggest a cheaper option. Now ask “what’s the weather in 5 cities?’’ — that’s five *independent* calls that should all fire at once. One task needs sequencing and a branch; the other needs parallelism. **Orchestration** is the layer that decides, for any task, which tools run together, which run in order, and which run only if a condition holds — and that recovers when one of them fails halfway through.

### 🎯 What you will be able to do after this lesson

- **Build** the three orchestration shapes — parallel (`asyncio.gather`), sequential chains, and conditional routing — and a small DAG executor that parallelizes independent nodes automatically.

- **Compare** agent architectures (single-agent, router, planner-executor, orchestrator-workers) and pick the simplest one that fits.

- **Operate** the agentic **ReAct** loop (Thought → Action → Observation) with a step cap, plus state management: token budgets, checkpointing, and compaction.

- **Evaluate** production concerns — saga + compensating transactions and circuit breakers for effectful chains, and OpenTelemetry tracing / cost attribution across the 2026 framework landscape.

> 📦 **Info**
>
> ✅ Before you startThis builds directly on **Lesson 10.1** (the tool-use loop: the model requests, your code runs) and **Lesson 10.2** (schema design). Here the model calls MANY tools across one task. You should be comfortable with the tool-use loop and basic `async`/`await`.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🎧 **Analogy**
>
> **Orchestration is conducting an orchestra.** Some instruments play at the same time (parallel); some enter one after another (sequential); the conductor cues who plays next based on what just happened (conditional). The **score is the DAG** — it encodes which parts overlap and which must wait. **Where it breaks down:** a real conductor never lets a section wait forever on a cue that never comes — a broken dependency (a tool that failed) must be handled (retry or compensate), not silently skipped.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Just run all the tools.”** Tools have a dependency graph. Independent ones parallelize; dependent ones must be sequenced. Running a dependent tool before its input exists produces garbage.
> - **“An agent loop always finishes.”** It doesn’t. Without a step cap it can loop forever. Bound the iterations.

> 💡 **Info**
>
> ⚠️ Anti-patternRetrying a failed **effectful** chain without compensation. The flight booked, the hotel call failed, so you retry the whole chain — and now you have booked TWO flights. Effectful chains need idempotent calls (so a retry cannot double-book) plus rollback (a saga) — not blind retry.

---

## 🎯 Concept 1: Parallel Execution

### Parallel Execution

Independent tool calls should run concurrently. Wall-clock becomes the slowest call, not the sum.

The first and biggest win is free: when tool calls don’t depend on each other, fire them all at once. Five weather lookups that each take ~50ms cost ~250ms in a naive loop but ~50ms in parallel — the wall-clock collapses to the *slowest single call* because they overlap. In Python that’s `asyncio.gather`; the model itself can also request several tool calls in one turn (you saw parallel tool calls in 10.1). The catch: this only works for *independent* calls.

> ⏱️ **Analogy**
>
> It’s a **restaurant kitchen**. Five independent dishes go on five burners at once, not one after another — dinner is ready in the time of the slowest dish, not the sum of all five. But you can’t plate before the sauce is done: a dependent step still waits.

Five independent weather calls, ~50ms each. Sequential loop vs `asyncio.gather` — wall-clock?

**📝 `01_parallel.py`** - *runnable*

In [ ]:
# PARALLEL vs SEQUENTIAL - independent tool calls should run concurrently.
import asyncio, time
import nest_asyncio; nest_asyncio.apply()   # allow asyncio.run() inside a notebook's running event loop

async def get_weather(city):                     # a stand-in tool with fixed latency
    await asyncio.sleep(0.05)
    return {"city": city, "temp_c": {"Hyderabad":34,"Mumbai":30,"Delhi":38,
                                     "Bangalore":28,"Chennai":33}[city]}

CITIES = ["Hyderabad","Mumbai","Delhi","Bangalore","Chennai"]

async def run_sequential():                      # one at a time: latencies ADD UP
    t = time.perf_counter(); out = []
    for c in CITIES:
        out.append(await get_weather(c))
    return out, (time.perf_counter()-t)*1000

async def run_parallel():                        # all at once: wall-clock = the SLOWEST call
    t = time.perf_counter()
    out = await asyncio.gather(*[get_weather(c) for c in CITIES])
    return list(out), (time.perf_counter()-t)*1000

async def main():
    _, seq_ms = await run_sequential()
    _, par_ms = await run_parallel()
    print(f"  {len(CITIES)} independent calls @ 50ms each:")
    print(f"    sequential ~= {len(CITIES)*50}ms  (5 x 50, latencies add up)")
    print(f"    parallel   ~= {round(par_ms/50)*50}ms   (one round-trip, = the slowest call)")
    print(f"    speedup    ~= {len(CITIES)}x")
asyncio.run(main())

# Output:
#   5 independent calls @ 50ms each:
#     sequential ~= 250ms  (5 x 50, latencies add up)
#     parallel   ~= 50ms   (one round-trip, = the slowest call)
#     speedup    ~= 5x

- `run_sequential` awaits each call in turn — latencies ADD UP to ~250ms (5 x 50).
- `run_parallel` fires all five with `asyncio.gather` — wall-clock is ~the slowest call (~50ms).
- The ~5x speedup is the number of INDEPENDENT calls; it does not apply to a dependent chain.
- Tradeoff: parallelism helps only when calls are independent; a dependency forces you back to sequencing (Step 2).

#### 💡 What just happened

⚡ What just happened? Parallelism is the cheapest orchestration win. **Independent** tool calls should always overlap — wall-clock becomes the slowest call, not the sum. The moment one call needs another’s output, you cannot parallelize it; that is the dependency graph of the next step.

- Play the sequential timeline, then the parallel one
- Slide the call count and watch wall-clock diverge

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: The Task DAG

### The Task DAG

Sequential and conditional, unified: a dependency graph. Run each node the moment its inputs are ready.

Sequential chains (A then B) and conditional routing (branch on a result) are the same idea seen through one lens: a **DAG** — a directed acyclic graph where each node names the tools it depends on. The executor is simple: repeatedly run every node whose dependencies are all done. Independent nodes land in the same “wave’’ and run in parallel; dependent nodes wait. The number of waves is the **critical path** — the longest dependency chain — and that, not the total node count, is your wall-clock in hops. This one executor gives you parallel, sequential, and conditional for free.

> 🏗️ **Analogy**
>
> It’s a **construction schedule**. Plumbing and wiring happen at once (independent); drywall waits for both; paint waits for drywall. The critical path is the longest chain of “this-before-that’’ — shortening anything OFF the critical path doesn’t make the house finish sooner.

A DAG: search + get_gst (independent), price needs both, then check_budget, then book. How many sequential hops?

**📝 `02_dag.py`** - *runnable*

In [ ]:
# THE TASK DAG - each node names the tools it depends on; run a node when its deps are done.
DAG = {
    "search_courses": [],                                  # independent
    "get_gst_rate":   [],                                  # independent (runs WITH search)
    "price_with_gst": ["search_courses", "get_gst_rate"],  # needs BOTH
    "check_budget":   ["price_with_gst"],                  # conditional gate
    "book":           ["check_budget"],                    # only if budget ok
}

def run_dag(dag):
    done, waves = set(), []
    while len(done) < len(dag):
        ready = [n for n in dag if n not in done and all(d in done for d in dag[n])]
        if not ready:                                      # nothing can advance -> a cycle or a missing dependency
            raise ValueError(f"stuck: cyclic or unsatisfiable deps in {set(dag) - done}")
        waves.append(ready)                                # a wave runs in PARALLEL
        for n in ready: done.add(n)
    return waves

waves = run_dag(DAG)
print("  Execution waves (each wave parallel; waves run in sequence):")
for i, w in enumerate(waves, 1):
    print(f"    wave {i}: {w}")
print(f"  Critical path = {len(waves)} hops (not {len(DAG)} - independent nodes overlap).")

# Output:
#   Execution waves (each wave parallel; waves run in sequence):
#     wave 1: ['search_courses', 'get_gst_rate']
#     wave 2: ['price_with_gst']
#     wave 3: ['check_budget']
#     wave 4: ['book']
#   Critical path = 4 hops (not 5 - independent nodes overlap).

- Each node lists its dependencies; `run_dag` runs every node whose deps are all done.
- Wave 1 is `search_courses` + `get_gst_rate` together — independent, so they parallelize.
- `price_with_gst` waits for both; `check_budget` is the conditional gate; `book` is last.
- Critical path = 4 waves, not 5 nodes — the win is the overlap. When to use: any task with a mix of independent and dependent tools.

#### 💡 What just happened

⚡ What just happened? A DAG unifies all three shapes. **Independent nodes parallelize; dependent nodes sequence; a conditional node is just a gate on a dependency.** The critical path (longest chain) is your true wall-clock — optimizing a node off the critical path buys nothing. Every orchestration framework (LangGraph, Agents SDK) is a DAG executor under the hood. One guard: a graph with a CYCLE (or a missing dependency) never terminates, so the executor must detect a stuck wave and raise — the DAG version of the ReAct step cap.

- Toggle node dependencies and Play the executor
- Watch independent nodes light up together; see the critical path

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: The ReAct Loop

### The ReAct Loop

Thought → Action → Observation, repeated. This is what turns tool use into multi-step problem solving.

A DAG is great when you know the plan up front. But often the agent must *discover* the next step from what it just saw. **ReAct** (Reason + Act) is the loop that does this: the model produces a *Thought*, takes an *Action* (a tool call), reads the *Observation* (the result), and loops — until it has the answer or hits a **step cap**. The cap is non-negotiable: without it, a model can loop forever, re-calling the same tool. **Reflexion** adds a twist — after a failure the agent writes a short lesson to memory and retries smarter.

> 🧭 **Analogy**
>
> It’s a **detective at a crime scene**. Look at a clue (observation), form a hunch (thought), investigate one thing (action), see what turns up, repeat. The detective doesn’t plan every step in advance — each observation decides the next move. And a good detective knows when to stop chasing a dead end (the step cap).

A ReAct agent keeps re-calling the same failing tool. What stops it from running forever?

**📝 `03_react.py`** - *runnable*

In [ ]:
# THE ReAct LOOP - Thought -> Action -> Observation, repeat until done or a step cap fires.
TOOLS = {
    "search":        lambda q: "GenAI Bootcamp INR 14999",
    "extract_price": lambda s: 14999,
    "add_gst":       lambda p: round(int(p)*1.18, 2),
}
def mock_reason(scratch):                         # a deterministic 'model' so the loop reproduces
    if "price"  not in scratch: return ("I need the price", "search", "cost?")
    if "amount" not in scratch: return ("Extract the number", "extract_price", scratch["price"])
    if "total"  not in scratch: return ("Add 18% GST", "add_gst", scratch["amount"])
    return ("I have the answer", "finish", scratch["total"])

def react(max_steps=6):
    scratch = {}
    for s in range(1, max_steps+1):
        thought, action, arg = mock_reason(scratch)
        if action == "finish":
            print(f"  step {s}: THOUGHT: {thought} -> ANSWER: total = {arg}"); return arg
        obs = TOOLS[action](arg)
        print(f"  step {s}: THOUGHT: {thought} | ACTION: {action} | OBS: {obs}")
        scratch |= {"search":{"price":obs}, "extract_price":{"amount":obs},
                    "add_gst":{"total":obs}}[action]
    print(f"  step cap ({max_steps}) hit -> stop; never loop forever"); return None
react()

# Output:
#   step 1: THOUGHT: I need the price | ACTION: search | OBS: GenAI Bootcamp INR 14999
#   step 2: THOUGHT: Extract the number | ACTION: extract_price | OBS: 14999
#   step 3: THOUGHT: Add 18% GST | ACTION: add_gst | OBS: 17698.82
#   step 4: THOUGHT: I have the answer -> ANSWER: total = 17698.82

- Each step prints THOUGHT, ACTION, and OBSERVATION — the loop’s three beats.
- `mock_reason` stands in for the model: it decides the next action from the scratchpad so the run reproduces.
- The loop ends when the action is `finish` — or, crucially, when `max_steps` is hit.
- Reflexion (not shown) adds a self-critique: on failure, write a lesson to memory and retry — the tradeoff is extra tokens for a smarter second attempt.

#### 💡 What just happened

⚡ What just happened? ReAct is the agentic loop: **Thought → Action → Observation**, discovering the plan as it goes. The step cap is the one line that separates a robust agent from an infinite loop. This loop is the heart of every agent framework and is what Module 11 builds into fully autonomous agents.

- Step through Thought -> Action -> Observation
- Watch the step cap fire, then a Reflexion retry

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Agent Architectures

### Agent Architectures

Single-agent, router, planner-executor, orchestrator-workers. Start simple; add structure only when it earns its keep.

How many agents does a task need? Anthropic’s guidance is blunt: **start with one**. A **single agent + tools** handles most tasks. Add a **router** (a cheap classifier that sends the request to the right specialist, and the right model tier — Haiku for easy, Sonnet for hard) when you have distinct request types. Add a **planner-executor** (one agent plans, another executes) when a task needs upfront decomposition. Add **orchestrator-workers** (a lead agent fans work out to parallel sub-agents and synthesizes) only for genuinely parallel, open-ended research. Each rung adds coordination cost, latency, and tokens.

> 🏥 **Analogy**
>
> It’s **staffing a company**. A one-person startup does everything (single agent). Growth adds a receptionist who routes calls (router), then a manager who plans and delegates (planner-executor), then departments with a director coordinating them (orchestrator-workers). You don’t hire a 200-person org to answer one email.

“The app crashes on login.’’ Following “start simple,’’ the right first move?

**📝 `04_architectures.py`** - *runnable*

In [ ]:
# AGENT ARCHITECTURES - start simple; add structure only when one agent cannot cope.
def router(query):                                # a cheap classifier -> specialist + model tier
    q = query.lower()
    if any(w in q for w in ("refund","cancel","order")): return ("orders_agent", "haiku (cheap)")
    if any(w in q for w in ("bug","error","crash")):     return ("eng_agent",    "sonnet (capable)")
    return ("general_agent", "haiku (cheap)")

def planner(goal):                                # decompose a complex goal into an ordered plan
    return ["search options", "compare on price", "check budget", "book or suggest cheaper"]

for q in ["I want a refund for order 42", "the app crashes on login"]:
    agent, tier = router(q)
    print(f"  ROUTER: {q!r:34s} -> {agent:14s} [{tier}]")
print("  PLANNER decomposes 'plan my Goa trip':")
for i, task in enumerate(planner("goa"), 1):
    print(f"    {i}. {task}")

# Output:
#   ROUTER: 'I want a refund for order 42'   -> orders_agent   [haiku (cheap)]
#   ROUTER: 'the app crashes on login'       -> eng_agent      [sonnet (capable)]
#   PLANNER decomposes 'plan my Goa trip':
#     1. search options
#     2. compare on price
#     3. check budget
#     4. book or suggest cheaper

- `router` is a cheap keyword classifier — it picks a specialist AND a model tier (Haiku vs Sonnet).
- `planner` decomposes a complex goal into an ordered task list (the planner-executor pattern).
- Orchestrator-workers (not shown) is a lead agent fanning out to parallel sub-agents — Anthropic’s research system and Claude Code use it.
- Tradeoff: each rung up the ladder adds coordination cost, latency, and tokens — use the simplest one that works.

#### 💡 What just happened

⚡ What just happened? Architectures form a ladder: **single-agent → router → planner-executor → orchestrator-workers**. Climb it only when the current rung genuinely cannot do the job. Model routing (cheap model for easy, capable for hard) is often the single highest-leverage move. Module 11 goes deep on autonomous multi-agent design.

- Pick single / router / planner / orchestrator-workers
- Watch the message flow reshape for each

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: State Management

### State Management

Short, long, and episodic memory. Keep the window under a token budget; checkpoint every step.

A multi-step run accumulates state, and state has three lifetimes. **Short-term** is the current conversation — it lives in the context window and must be kept under a **token budget**. **Long-term** is a memory store (a vector DB or files) the agent reads and writes across sessions. **Episodic** is summaries of past runs the agent learns from. Two production habits make this robust: **compaction** (when the window fills, summarize the oldest turns instead of dropping them) and **checkpointing** (persist state after every step so a crash *resumes* rather than restarts).

> 📝 **Analogy**
>
> It’s a **chef’s notes during a long service**. The active ticket rail is short-term memory (small, always in view); the recipe binder is long-term (looked up when needed); the “what went wrong last Friday’’ notes are episodic. When the rail fills, you summarize old tickets rather than lose them — and you write down where you are so a power cut doesn’t restart the whole service.

A long conversation is about to overflow the context window. Production-correct handling?

**📝 `05_state.py`** - *runnable*

In [ ]:
# STATE - keep the context window under a token budget; checkpoint after every step.
def tok(msg): return max(1, len(msg)//4)          # ~4 chars per token

class Conversation:
    def __init__(self, budget=40):
        self.turns=[]; self.budget=budget; self.summary=None; self.checkpoint=None
    def total(self):
        return (tok(self.summary) if self.summary else 0) + sum(tok(t) for t in self.turns)
    def add(self, msg):
        self.turns.append(msg)
        if self.total() > self.budget:            # COMPACT: fold oldest turns into a summary
            half = len(self.turns)//2
            self.summary = f"[summary of {half} earlier turns]"; self.turns = self.turns[half:]
        self.checkpoint = {"summary": self.summary, "turns": list(self.turns)}   # persist EVERY step

c = Conversation(budget=40)
for i in range(1, 9):
    c.add(f"user turn number {i} with some content here")
print(f"  after 8 turns: {len(c.turns)} live turns + summary={c.summary!r}")
print(f"  token total = {c.total()} (budget {c.budget}) -> compaction kept us under budget")
restored = Conversation()
restored.summary = c.checkpoint["summary"]; restored.turns = c.checkpoint["turns"]
print(f"  crash + resume -> {len(restored.turns)} turns restored, not 0 (resume, not restart)")

# Output:
#   after 8 turns: 2 live turns + summary='[summary of 2 earlier turns]'
#   token total = 27 (budget 40) -> compaction kept us under budget
#   crash + resume -> 2 turns restored, not 0 (resume, not restart)

- `tok` approximates tokens as ~4 chars each; `total` sums the summary plus live turns.
- When `total` exceeds the budget, the oldest half of turns is folded into a `summary` — that is compaction.
- `checkpoint` is saved after every `add`, so `restored` resumes with the turns intact, not zero.
- Tradeoff: compaction trades detail for staying under budget — summarize, do not silently drop, or the agent forgets the task.

#### 💡 What just happened

⚡ What just happened? State has three lifetimes — **short-term** (window, budgeted), **long-term** (memory store), **episodic** (past-run summaries). The framework gives you the checkpointer; YOU decide the budget and what to compact. Checkpointing is what makes a long agent run survive a crash. Module 14 covers evaluating agent memory quality.

- Slide the turn count and watch the window fill
- See compaction trigger and a checkpoint resume

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Error Handling for Chains

### Error Handling for Chains

A mid-chain failure leaves the world half-changed. Saga + compensating transactions roll it back; circuit breakers stop the bleeding.

This is the part demos skip and production cannot. When an **effectful** chain fails halfway — the flight booked, the hotel call 503’d — the world is now inconsistent, and retrying the whole chain double-books. The **saga pattern** pairs every forward action with a **compensating transaction** (`cancel_flight` undoes `book_flight`); on failure you run the compensations in reverse to roll back to a clean state. A **circuit breaker** stops you from hammering a tool that keeps failing: after N failures it “opens’’ and fails fast, cools down, then half-opens to test. Three complementary layers protect an effectful chain: make each effectful call **idempotent** (pass a stable request/idempotency key so a retry is de-duplicated into a no-op instead of a second booking) so retries are SAFE; use a **saga** to clean up after a later step fails; and a **circuit breaker** to stop hammering a dead tool. Round it out with retries using exponential backoff + jitter (a small random delay so retries do not all fire at once) for transient errors, and per-tool timeouts.

> 🏦️ **Analogy**
>
> It’s a **bank transfer**. Debit one account, credit the other — if the credit fails, the bank does NOT leave your money vanished; it reverses the debit (the compensating transaction). And if a payment rail keeps timing out, the bank stops routing to it (opens the circuit) instead of retrying into the void.

A chain books the flight, then the hotel call fails. Best handling?

**📝 `06_saga.py`** - *runnable*

In [ ]:
# SAGA + CIRCUIT BREAKER - an effectful chain that fails mid-way must ROLL BACK, not retry.
LEDGER = []
def book_flight(): LEDGER.append("flight"); return "flight booked"
def book_hotel():  raise RuntimeError("hotel API 503")      # inject a mid-chain failure
def book_car():    LEDGER.append("car");    return "car booked"
COMPENSATE = {"flight": lambda: LEDGER.remove("flight"), "car": lambda: LEDGER.remove("car")}

def saga(steps):
    done = []
    for name, forward in steps:
        try:
            forward(); done.append(name); print(f"    OK   {name}")
        except Exception as e:
            print(f"    FAIL {name}: {e} -> roll back {done[::-1]}")
            for d in reversed(done):
                COMPENSATE[d](); print(f"      compensated {d}")
            return False
    return True

print("  Saga (flight -> hotel -> car):")
saga([("flight", book_flight), ("hotel", book_hotel), ("car", book_car)])
print(f"    final ledger = {LEDGER}   (empty = no half-booked trip)")

class CircuitBreaker:
    def __init__(self, threshold=3):
        self.fails = 0; self.threshold = threshold; self.state = "closed"
    def call(self, fn):
        if self.state == "open": return "SKIPPED (circuit open)"
        try:
            r = fn(); self.fails = 0; return r
        except Exception:
            self.fails += 1
            if self.fails >= self.threshold: self.state = "open"
            return f"fail {self.fails}/{self.threshold}" + (" -> OPEN" if self.state=="open" else "")

cb = CircuitBreaker()
print("  Circuit breaker on a flaky tool:")
for i in range(5):
    print(f"    call {i+1}: {cb.call(book_hotel)}")

# Output:
#   Saga (flight -> hotel -> car):
#     OK   flight
#     FAIL hotel: hotel API 503 -> roll back ['flight']
#       compensated flight
#     final ledger = []   (empty = no half-booked trip)
#   Circuit breaker on a flaky tool:
#     call 1: fail 1/3
#     call 2: fail 2/3
#     call 3: fail 3/3 -> OPEN
#     call 4: SKIPPED (circuit open)
#     call 5: SKIPPED (circuit open)

- `saga` runs each forward action; on failure it runs `COMPENSATE` for each completed step in REVERSE order.
- The hotel fails after the flight booked, so `compensated flight` runs — final ledger is empty (no half-booked trip).
- `CircuitBreaker` opens after 3 failures and then SKIPS further calls — it fails fast instead of hammering a dead tool.
- Tradeoff: compensation needs an undo for every effectful action — design it up front, or a partial failure strands the user.

#### 💡 What just happened

⚡ What just happened? Effectful chains need rollback, not retry. The **saga pattern** (forward action + compensating transaction) leaves the world consistent after a mid-chain failure; a **circuit breaker** stops you hammering a broken tool. Retries with backoff + per-tool timeouts round out the layer. This is the difference between a demo and a system real users trust.

- Inject a mid-chain failure and watch compensations roll back
- Trip the circuit breaker: open, cool down, half-open

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Agent Frameworks (2026)

### Agent Frameworks (2026)

The same loop, four ecosystems. Know the current APIs and what each is for.

You can build all of the above by hand — and for simple agents you should. Frameworks earn their keep when you need durable state, human-in-the-loop, or multi-agent coordination. The 2026 landscape: **LangGraph v1.0** models an agent as a `StateGraph` with checkpointers (`InMemorySaver` / `SqliteSaver` / `PostgresSaver`) and human-in-the-loop via `interrupt()`; its prebuilt `create_react_agent` is now deprecated in favor of `create_agent` in the `langchain` package. The **OpenAI Agents SDK** gives you `Agent` + `Runner` (which runs the tool loop and switches agents on handoff) + `Sessions`, and *replaces the experimental Swarm*. The **Claude Agent SDK** runs agents on the same harness as Claude Code, MCP-native. **CrewAI** is the fastest path to a role-based prototype; **AutoGen** is community-maintained now, with Microsoft steering new enterprise work to the **Microsoft Agent Framework**.

A colleague’s 2026 agent imports `Swarm` and LangGraph’s `create_react_agent`. What do you tell them?

**📝 `07_frameworks.py current`** - *APIs*

**`07_frameworks.py`** (illustrative APIs - needs the libraries installed and an API key; not run here)

```python
# THE SAME ReAct AGENT IN TWO 2026 FRAMEWORKS (illustrative; needs the libraries + a key).
# Minimal example only - production code adds tracing, retries, and human-in-the-loop.

# --- LangGraph v1.0 (durable state via a checkpointer; create_agent replaces create_react_agent) ---
from langchain.agents import create_agent          # NOT langgraph.prebuilt.create_react_agent (deprecated)
from langgraph.checkpoint.memory import InMemorySaver
agent = create_agent(model="claude-sonnet-4-6", tools=[get_weather, add_gst],
                     checkpointer=InMemorySaver())  # SqliteSaver / PostgresSaver in production
out = agent.invoke({"messages": [{"role":"user","content":"weather in 5 cities, then the total"}]},
                   config={"configurable": {"thread_id": "trip-1"}})   # thread_id = resumable state

# --- OpenAI Agents SDK (Runner runs the loop + handoffs; replaces the experimental Swarm) ---
from agents import Agent, Runner
weather_agent = Agent(name="weather", instructions="Report weather.", tools=[get_weather])
planner_agent = Agent(name="planner", instructions="Plan a trip.", handoffs=[weather_agent])
result = Runner.run_sync(planner_agent, "weather in 5 cities, then the total")   # Sessions = auto history

# Output: (illustrative - runs with the frameworks installed and a key set)
```


- **LangGraph:** `create_agent` (in `langchain`) replaces the deprecated `create_react_agent`; a checkpointer + `thread_id` gives durable, resumable state.
- **OpenAI Agents SDK:** `Agent` + `Runner` run the loop and handle handoffs; it replaces the deprecated Swarm; Sessions manage history.
- Both are illustrative — they need the libraries installed and an API key; the deterministic blocks above run without either.
- When to use a framework vs hand-rolled: reach for one when you need durable checkpointing, human-in-the-loop, or multi-agent handoffs; otherwise plain code is simpler.

#### 💡 What just happened

⚡ What just happened? Frameworks implement the same DAG + ReAct loop you just built by hand, adding durable state, HITL, and multi-agent coordination. The 2026 currency that matters: **Swarm → OpenAI Agents SDK**, **create_react_agent → create_agent**, and LangGraph checkpointers for resumability. Prefer plain code until a framework genuinely earns the abstraction.

---

## 🎯 Concept 8: Observability & India

### Observability & India

You cannot debug a 12-step run from stdout. Trace every step; attribute cost per tool.

A single tool call you can eyeball. A twelve-step, multi-agent run is invisible without **tracing**. The **OpenTelemetry GenAI semantic conventions** standardize a `gen_ai.*` span namespace — operation, model, prompt/completion, tokens, cost, tool calls, finish reason — plus dedicated agent spans. Emit one span per tool call and you can answer “which agent ran, which tools, which handoffs, and what did each step cost?’’ **Langfuse** and **LangSmith** are the common backends; Datadog and New Relic ingest the conventions natively. **Per-tool cost attribution** is how you find the one expensive step in a chain. **For India:** the same trace gets a rupee column and a translate span per hop — **Sarvam** (OpenAI-compatible, so Agents-SDK-style code works with a `base_url` swap) plus **Bhashini** for 22-language STT/translate/TTS, keeping the translate-first pattern from 10.1 at every hop.

Your multi-agent run is slow and expensive but you don’t know which step is the culprit. First move?

**📝 `08_observability.py`** - *runnable*

In [ ]:
# OBSERVABILITY - emit OpenTelemetry gen_ai spans; attribute cost per tool to find the expensive step.
PRICES = {"haiku": (1.0, 5.0), "sonnet": (3.0, 15.0)}     # $ per 1M tokens; illustrative current-tier rates (they move)

def span(tool, model, tin, tout):
    pin, pout = PRICES[model]
    cost = (tin*pin + tout*pout) / 1_000_000
    return {"gen_ai.tool.name": tool, "gen_ai.request.model": model,
            "gen_ai.usage.input_tokens": tin, "gen_ai.usage.output_tokens": tout,
            "cost_usd": round(cost, 6)}

trace = [span("router","haiku",120,20), span("search","haiku",300,150),
         span("price_with_gst","haiku",200,40), span("book","sonnet",400,120)]
print("  Trace (one gen_ai span per tool call):")
for s in trace:
    print(f"    {s['gen_ai.tool.name']:16s} {s['gen_ai.request.model']:7s} "
          f"in={s['gen_ai.usage.input_tokens']:4d} out={s['gen_ai.usage.output_tokens']:4d} "
          f"${s['cost_usd']}")
print(f"  total run cost = ${round(sum(s['cost_usd'] for s in trace), 6)}  (per-tool -> find the pricey step)")

# Output:
#   Trace (one gen_ai span per tool call):
#     router           haiku   in= 120 out=  20 $0.00022
#     search           haiku   in= 300 out= 150 $0.00105
#     price_with_gst   haiku   in= 200 out=  40 $0.0004
#     book             sonnet  in= 400 out= 120 $0.003
#   total run cost = $0.00467  (per-tool -> find the pricey step)

- Each `span` uses the OTel `gen_ai.*` attribute names (tool, model, input/output tokens).
- Cost is computed per span from token counts and per-model prices — so you can see the `book` step on Sonnet dominates.
- Summing spans gives the total run cost, attributed per tool — that is how you find the expensive step.
- India: add a rupee column (about x83) and a translate span per hop (Sarvam / Bhashini); per-tool cost matters more where budgets are tighter.

#### 💡 What just happened

⚡ What just happened? Observability is not optional for multi-step agents. **OTel `gen_ai` spans** make a run reconstructable and cost-attributable; per-tool cost finds the expensive step. The India stack (Sarvam + Bhashini + translate-first) rides the same trace with a rupee column. Module 14 (LLMOps) goes deep on agent evals and cost/perf at scale.

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Orchestration is one idea in layers. **Parallel** calls overlap (Step 1); a **DAG** unifies parallel, sequential, and conditional and runs each node when its inputs are ready (Step 2); the **ReAct** loop discovers the plan step by step when you can’t know it up front (Step 3). Choose the simplest **architecture** that fits (Step 4), keep **state** under a token budget and checkpoint it (Step 5), and make effectful chains recover with **saga + compensation** and circuit breakers (Step 6). Frameworks (Step 7) implement all of this; **observability** (Step 8) makes it debuggable and cost-attributable. Master these and you can orchestrate any set of tools into a system real users trust.

> 📦 **Info**
>
> ➡️ Where this goes nextNext up, in Lesson 10.4 you build real MCP servers and clients that expose these tools over a standard protocol. Fully autonomous multi-step agents (planning, tool-use-at-scale, agent evaluation) come back in Module 11, and cost, tracing, and error analysis at production scale arrive in Module 14. The reference for the patterns here is Anthropic’s [Building Effective Agents](https://www.anthropic.com/research/building-effective-agents).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Multi-ToolOrchestration**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-10_3.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-10.3.html` - regenerate this notebook whenever the source HTML is updated.*
